In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

file_path = '/content/drive/MyDrive/fake reviews dataset.csv'

df = pd.read_csv(file_path)

print(df.shape)
df.head()

(40432, 4)


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [3]:
import pandas as pd
import re
import html

file_path = "/content/drive/MyDrive/fake reviews dataset.csv"
df = pd.read_csv(file_path)

def clean_text(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_text"] = df["text_"].apply(clean_text)

In [4]:
df.head()

,category,rating,label,text_,clean_text
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor...","Love this! Well made, sturdy, and very comfort..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I...","love it, a great upgrade from the original. I'..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...","Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,Very nice set. Good quality. We have had the s...


In [5]:
df["label_id"] = df["label"].map({
    "CG": 0,
    "OR": 1
})

df[["label", "label_id"]].head()

,label,label_id
0,CG,0
1,CG,0
2,CG,0
3,CG,0
4,CG,0


In [6]:
unique_df = df.drop_duplicates(
    subset="clean_text"
).copy()

print("Original rows:", len(df))
print("Unique reviews:", len(unique_df))
print("Removed repeated rows:", len(df) - len(unique_df))

Original rows: 40432
Unique reviews: 40393
Removed repeated rows: 39


In [7]:
print(unique_df["label"].value_counts())

print("\nLabel proportions:")
print(unique_df["label"].value_counts(normalize=True) * 100)

label
OR    20215
CG    20178
Name: count, dtype: int64

Label proportions:
label
OR    50.0458
CG    49.9542
Name: proportion, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

train_val, test_df = train_test_split(
    unique_df,
    test_size=0.15,
    random_state=42,
    stratify=unique_df["label_id"]
)

print("Train + Validation:", len(train_val))
print("Test:", len(test_df))

Train + Validation: 34334
Test: 6059


In [9]:
train_df, val_df = train_test_split(
    train_val,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_val["label_id"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 28275
Validation: 6059
Test: 6059
Total: 40393


In [10]:
for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} label counts:")
    print(split_df["label"].value_counts())

    print(f"\n{name} label percentages:")
    print(
        (split_df["label"].value_counts(normalize=True) * 100).round(2)
    )


Train label counts:
label
OR    14151
CG    14124
Name: count, dtype: int64

Train label percentages:
label
OR    50.05
CG    49.95
Name: proportion, dtype: float64

Validation label counts:
label
OR    3032
CG    3027
Name: count, dtype: int64

Validation label percentages:
label
OR    50.04
CG    49.96
Name: proportion, dtype: float64

Test label counts:
label
OR    3032
CG    3027
Name: count, dtype: int64

Test label percentages:
label
OR    50.04
CG    49.96
Name: proportion, dtype: float64


In [11]:
# Check whether cleaned reviews overlap between splits

train_texts = set(train_df["clean_text"])
val_texts = set(val_df["clean_text"])
test_texts = set(test_df["clean_text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [12]:
import os

output_dir = "/content/drive/MyDrive/fake review detection system/data"

os.makedirs(output_dir, exist_ok=True)

print("Folder ready:", output_dir)

Folder ready: /content/drive/MyDrive/fake review detection system/data


In [13]:
train_df.to_csv(
    os.path.join(output_dir, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(output_dir, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(output_dir, "test.csv"),
    index=False
)

print("All three datasets saved successfully!")

All three datasets saved successfully!


In [14]:
for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = os.path.join(output_dir, filename)
    saved_df = pd.read_csv(path)

    print(f"{filename}: {saved_df.shape}")

train.csv: (28275, 6)
validation.csv: (6059, 6)
test.csv: (6059, 6)
